# Hindi model: reasoning finetuning (Colab)

Finetunes the Phase 2 pretrained hindi checkpoint on the synthetic reasoning dataset (`hindi/data/reasoning/`), via LoRA on `q_proj`/`v_proj` (rank 8, alpha 16, per `docs-phase-3/lora_rank_sweep_result.md`). Assumes the `phase-3` branch has already been pushed to GitHub, the reasoning dataset is already committed there, and the Phase 2 pretrained checkpoint + its `val.bin` (for the pretrain-val PPL forgetting signal) are present on Drive at the same `lma_phase2_checkpoints` folder Phase 2 used.

Runs one seed per cell-6 execution. Rerun cell 6 with a different `SEED` (see `SEEDS` list) to produce the next seed's checkpoint -- per-seed resumability: a seed whose finetuned checkpoint already exists on Drive is skipped, not retrained, so reconnecting after a disconnect and rerunning from the top picks up at the next un-run seed automatically.

Early stopping (patience on reasoning-task validation loss, with a pretrain-val PPL forgetting-threshold override) picks the actual stopping epoch per run -- see `finetune.py`'s `finetune()` docstring. No manual per-epoch checkpoint inspection needed.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os
from google.colab import userdata

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
REPO_URL = f'https://{GITHUB_TOKEN}@github.com/CL3-410/individual-project-abhinavborah'
REPO_DIR = '/content/individual-project-abhinavborah'

if not os.path.exists(REPO_DIR):
    !git clone -b phase-3 {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git checkout phase-3 && git pull


In [ ]:
!pip install -q sentencepiece


In [ ]:
import json
import sys
from pathlib import Path

import torch

LANGUAGE = 'hindi'
REPO_ROOT = Path(REPO_DIR) / LANGUAGE

sys.path.insert(0, str(REPO_ROOT / 'model'))
sys.path.insert(0, str(REPO_ROOT / 'train'))
from finetune import (
    finetune, freeze_non_lora_params, inject_lora, load_pretrained_weights,
    load_reasoning_examples, merge_lora_to_plain, pretrain_val_ppl,
)
from model import DecoderLM

import sentencepiece as spm
import numpy as np

# Phase 2's pretrained checkpoint + val.bin (for the pretrain-val PPL forgetting
# signal) already live here from Phase 2 -- read-only, never modified.
PHASE2_CHECKPOINT_DIR = Path('/content/drive/MyDrive/lma_phase2_checkpoints')
PRETRAINED_CKPT = PHASE2_CHECKPOINT_DIR / f'{LANGUAGE}_checkpoint.pt'
PRETRAIN_VAL_BIN = PHASE2_CHECKPOINT_DIR / f'{LANGUAGE}_val.bin'

# Phase 3 finetuned checkpoints go in their own Drive folder, separate from
# the pretrained ones -- spec wants separate, additional checkpoints, not an
# overwrite of the pretrained artifact.
FINETUNE_CHECKPOINT_DIR = Path('/content/drive/MyDrive/lma_phase3_checkpoints')
FINETUNE_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR = FINETUNE_CHECKPOINT_DIR / 'logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)

REASONING_DIR = REPO_ROOT / 'data' / 'reasoning'
TOKENIZER_PATH = REPO_ROOT / 'tokenizer' / f'{LANGUAGE}_bpe_8000.model'
CONFIG_PATH = REPO_ROOT / 'configs' / 'model_config.json'

# Picked via docs-phase-3/lora_rank_sweep_result.md, confirmed independently
# on both languages' real data.
LORA_RANK = 8
LORA_ALPHA = 16

BATCH_SIZE = 32
MAX_EPOCHS = 40  # generous ceiling; early stopping picks the real stop point
EARLY_STOPPING_PATIENCE = 5
PEAK_LR = 1e-4  # within the CeADAR guide's cited 1e-4 to 2e-4 range
MIN_LR = 1e-5
WARMUP_STEPS = 10

SEEDS = [0, 1, 2]  # Lec09's multi-seed requirement: 2-3 seeds per language

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {device}')


In [ ]:
with open(CONFIG_PATH, encoding='utf-8') as f:
    full_config = json.load(f)
model_config = full_config['model']
opt_config = full_config['optimizer']

sp = spm.SentencePieceProcessor()
sp.load(str(TOKENIZER_PATH))

train_examples = load_reasoning_examples(REASONING_DIR / 'train.jsonl')
val_examples = load_reasoning_examples(REASONING_DIR / 'val.jsonl')
print(f'train: {len(train_examples)}, val: {len(val_examples)}')

assert PRETRAINED_CKPT.exists(), f'pretrained checkpoint not found at {PRETRAINED_CKPT}'
assert PRETRAIN_VAL_BIN.exists(), f'pretrain val.bin not found at {PRETRAIN_VAL_BIN}'
pretrain_val_data = torch.from_numpy(
    np.memmap(PRETRAIN_VAL_BIN, dtype=np.uint16, mode='r').astype(np.int64)
)
print(f'pretrain val tokens: {len(pretrain_val_data):,}')

# Baseline (no finetune) pretrain-val PPL, for the forgetting-threshold check
# below and for the report's pretrained-vs-finetuned comparison.
baseline_model = DecoderLM(model_config)
load_pretrained_weights(PRETRAINED_CKPT, baseline_model)
baseline_model.to(device)
baseline_loss, baseline_ppl = pretrain_val_ppl(
    baseline_model, pretrain_val_data, model_config['context_length'], device
)
print(f'baseline (pretrained, no finetune) pretrain-val PPL: {baseline_ppl:.2f}')
del baseline_model


In [ ]:
results = []
for seed in SEEDS:
    ckpt_path = FINETUNE_CHECKPOINT_DIR / f'{LANGUAGE}_finetuned_seed{seed}.pt'
    log_path = LOG_DIR / f'{LANGUAGE}_finetune_seed{seed}_log.json'

    if ckpt_path.exists():
        print(f'seed {seed}: checkpoint already exists at {ckpt_path}, skipping')
        continue

    print(f'--- seed {seed} ---')
    torch.manual_seed(seed)
    model = DecoderLM(model_config)
    load_pretrained_weights(PRETRAINED_CKPT, model)
    inject_lora(model, rank=LORA_RANK, alpha=LORA_ALPHA)
    freeze_non_lora_params(model)
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        betas=(opt_config['beta1'], opt_config['beta2']), weight_decay=opt_config['weight_decay'],
    )

    # A real forgetting event (not just reasoning-task plateau) always overrides
    # patience -- threshold set at 15% above the no-finetune baseline PPL.
    ppl_forgetting_threshold = baseline_ppl * 1.15

    log = finetune(
        model, optimizer, sp, train_examples, val_examples, pretrain_val_data,
        num_epochs=MAX_EPOCHS, batch_size=BATCH_SIZE, context_length=model_config['context_length'],
        peak_lr=PEAK_LR, min_lr=MIN_LR, warmup_steps=WARMUP_STEPS,
        grad_clip_norm=opt_config['grad_clip_norm'], seed=seed,
        checkpoint_path=ckpt_path, config=full_config, device=device,
        early_stopping_patience=EARLY_STOPPING_PATIENCE,
        ppl_forgetting_threshold=ppl_forgetting_threshold,
    )

    with open(log_path, 'w', encoding='utf-8') as f:
        json.dump(log, f, indent=2)

    best = min(log, key=lambda e: e['reasoning_val_loss'])
    print(f'seed {seed}: stopped after {len(log)} epochs, best epoch {best["epoch"]}, '
          f'reasoning_val_loss={best["reasoning_val_loss"]:.4f}, pretrain_val_ppl={best["pretrain_val_ppl"]:.2f}')
    results.append({'seed': seed, 'best_epoch': best['epoch'], **{k: v for k, v in best.items() if k != 'epoch'}})

print()
print('--- summary across seeds ---')
for r in results:
    print(r)


In [ ]:
# Export: merge each seed's LoRA checkpoint into a plain DecoderLM checkpoint
# (spec's "same resume-capable format as pretraining" requirement, and the
# exact shape Phase 2's attention-analysis toolkit already expects).
for seed in SEEDS:
    lora_ckpt_path = FINETUNE_CHECKPOINT_DIR / f'{LANGUAGE}_finetuned_seed{seed}.pt'
    merged_ckpt_path = FINETUNE_CHECKPOINT_DIR / f'{LANGUAGE}_finetuned_seed{seed}_merged.pt'
    if merged_ckpt_path.exists():
        print(f'seed {seed}: merged checkpoint already exists, skipping')
        continue
    if not lora_ckpt_path.exists():
        print(f'seed {seed}: no LoRA checkpoint found, skipping')
        continue

    lora_model = DecoderLM(model_config)
    inject_lora(lora_model, rank=LORA_RANK, alpha=LORA_ALPHA)
    freeze_non_lora_params(lora_model)
    lora_optimizer = torch.optim.AdamW([p for p in lora_model.parameters() if p.requires_grad])
    from train import load_checkpoint
    load_checkpoint(lora_ckpt_path, lora_model, lora_optimizer, map_location=device)

    merged_model = merge_lora_to_plain(lora_model, model_config)
    merged_optimizer = torch.optim.AdamW(merged_model.parameters())
    from train import save_checkpoint
    save_checkpoint(merged_ckpt_path, merged_model, merged_optimizer, 0, full_config)
    print(f'seed {seed}: merged checkpoint written to {merged_ckpt_path}')
